**Importing the dataset**



In [10]:
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("rafsanmahmud333333/cifar-10-attacked-dataset")



Using Colab cache for faster access to the 'cifar-10-attacked-dataset' dataset.


**Data Preprocessing**

In [15]:
import os
import cv2
import pandas as pd
import numpy as np

# Set up paths to the dataset directory splits
base_path = os.path.join(path, "cifar10")
test_dir = os.path.join(base_path, "test")
clean_dir = os.path.join(test_dir, "clean")
fgsm_dir = os.path.join(test_dir, "fgsm")

# Load the test metadata to get the matching filenames and true labels
metadata_path = os.path.join(test_dir, "metadata_test(fgsm, bim, cANDw).csv")

if not os.path.exists(metadata_path):
    print(f"Error: Missing metadata file at {metadata_path}")
else:
    df_meta = pd.read_csv(metadata_path)

    # Filter the metadata sheet to only include row records belonging to the FGSM attack
    df_fgsm_meta = df_meta[df_meta['attack_type'] == 'fgsm']

    # Temporary lists to append the processed images and targets
    clean_images = []
    fgsm_images = []
    labels = []

    print(f"Running data loading and image preprocessing for {len(df_fgsm_meta)} FGSM samples...")

    # Iterate through the filtered metadata sheet to load matching image pairs synchronously
    for index, row in df_fgsm_meta.iterrows():
        # Extract just the filename from the end of the full path string
        full_path_str = row['image_path']
        filename = os.path.basename(full_path_str)

        # Pull the ground truth label column from the dataframe
        true_label = row['original_label']

        clean_img_path = os.path.join(clean_dir, filename)
        fgsm_img_path = os.path.join(fgsm_dir, filename)

        # Verify file existence to ensure data integrity during loading
        if os.path.exists(clean_img_path) and os.path.exists(fgsm_img_path):
            # Load images using OpenCV (Default format is BGR)
            bgr_clean = cv2.imread(clean_img_path)
            bgr_fgsm = cv2.imread(fgsm_img_path)

            # Convert color space from BGR to RGB for correct metric calculation
            rgb_clean = cv2.cvtColor(bgr_clean, cv2.COLOR_BGR2RGB)
            rgb_fgsm = cv2.cvtColor(bgr_fgsm, cv2.COLOR_BGR2RGB)

            # Enforce strict CIFAR-10 spatial dimension constraints (32x32)
            if rgb_clean.shape[:2] != (32, 32):
                rgb_clean = cv2.resize(rgb_clean, (32, 32), interpolation=cv2.INTER_AREA)
            if rgb_fgsm.shape[:2] != (32, 32):
                rgb_fgsm = cv2.resize(rgb_fgsm, (32, 32), interpolation=cv2.INTER_AREA)

            # Normalize pixel intensities from [0, 255] to [0.0, 1.0] for model evaluation
            norm_clean = rgb_clean.astype(np.float32) / 255.0
            norm_fgsm = rgb_fgsm.astype(np.float32) / 255.0

            clean_images.append(norm_clean)
            fgsm_images.append(norm_fgsm)
            labels.append(true_label)

    # Convert lists to NumPy arrays for easier matrix manipulation later
    X_clean = np.array(clean_images)
    X_fgsm = np.array(fgsm_images)
    y_test = np.array(labels)

    # Verification summary to check dataset tensor shapes
    print("\n--- Preprocessing Pipeline Verification ---")
    print(f"X_clean Matrix Shape : {X_clean.shape}")
    print(f"X_fgsm Matrix Shape  : {X_fgsm.shape}")
    print(f"y_test Target Shape  : {y_test.shape}")
    if len(X_clean) > 0:
        print(f"Pixel value bounds   : Min = {X_clean.min()}, Max = {X_clean.max()}")
        print(f"Total unique classes : {len(np.unique(y_test))}")
    else:
        print("⚠ Warning: No images were loaded. Check folder directory matching.")

Running data loading and image preprocessing for 500 FGSM samples...

--- Preprocessing Pipeline Verification ---
X_clean Matrix Shape : (500, 32, 32, 3)
X_fgsm Matrix Shape  : (500, 32, 32, 3)
y_test Target Shape  : (500,)
Pixel value bounds   : Min = 0.0, Max = 1.0
Total unique classes : 10
